In [1]:
import os
import json
import urllib.request
import importlib, pathlib, sys
from urllib.parse import urlparse
from datetime import datetime, timezone

import teehr
import pandas as pd
from teehr.evaluation.spark_session_utils import create_spark_session

from teehr import DeterministicMetrics as dm
from teehr import Signatures as s
from teehr import RowLevelCalculatedFields as rcf
from teehr import TimeseriesAwareCalculatedFields as tcf
from teehr import Bootstrappers as bs

from teehr.models.filters import TableFilter

from pyspark.sql import functions as F

from pyspark.sql import DataFrame

import copy
import time

teehr.__version__

'0.7.0'

In [2]:
# spark = create_spark_session(
#     start_spark_cluster=True,
#     executor_instances=64,
#     executor_memory="16g",
#     executor_cores=2,
#     aws_profile="default",
#     pod_template_path=pod_template_path,
#     update_configs={
#         "spark.sql.shuffle.partitions": 1024,
#         "spark.sql.adaptive.coalescePartitions.enabled": "false",
#         "spark.kubernetes.executor.annotation.cluster-autoscaler.kubernetes.io/safe-to-evict": "false",
#         "spark.executorEnv.TEEHR_BOOTSTRAP_ENGINE": "vectorized",
#         "spark.executor.memoryOverhead": "4g",
#     }
# )

spark = create_spark_session()

INFO:teehr.evaluation.spark_session_utils:🚀 Creating Spark session: TEEHR Evaluation
INFO:teehr.evaluation.spark_session_utils:✅ Spark local configuration successful!
INFO:teehr.evaluation.spark_session_utils:Setting Hadoop's default AWS credentials provider and AWS region
INFO:teehr.evaluation.spark_session_utils:🔑 Using AWS session token from boto3
INFO:teehr.evaluation.spark_session_utils:Configuring Iceberg catalogs...
INFO:teehr.evaluation.spark_session_utils:⚙️ All settings applied. Creating Spark session...
INFO:teehr.evaluation.spark_session_utils:🎉 Spark session created successfully!


In [3]:
spark.sql("USE iceberg.teehr")

DataFrame[]

In [4]:
spark.sql("""
SELECT *
FROM nwmd_metrics_by_location_v2 
LIMIT 10
""").show()

+-------------------+---------------------+------------------+---------+--------------------+------+----------+-------+----------------------+---------+----------+-----+------------------+-------------------+------------------+-----------+--------------------+--------------------+-------------------+-------------------+---------------------------+--------------------+-------------------------+----------------------+--------------------+------------------------+------------------------+--------------------------+--------------------------+---------------------------+---------------------------+---------------------------+---------------------------+--------------------------------------+--------------------------------------+------------------------+------------------------+------------------------------------+------------------------------------+---------------------------------+---------------------------------+------------------------------+------------------------------+------------

In [5]:
spark.sql("""
SELECT 
    count(*)
FROM fcst_joined_timeseries
WHERE configuration_name in ('nwm30_medium_range', 'nwm30_short_range')
""").show(truncate=False)


+----------+
|count(1)  |
+----------+
|9764115437|
+----------+



In [6]:
spark.sql("""
SELECT 
    count(*)
FROM nwmd_metrics_by_location_v2
GROUP BY configuration_name
""").show(truncate=False)

+--------+
|count(1)|
+--------+
|7378365 |
|2686017 |
+--------+



In [7]:
spark.sql("""
SELECT 
    configuration_name,
    collect_set(forecast_lead_time_bin) as forecast_lead_time_bins,
    collect_set(water_year) as water_years,
    collect_set(threshold) as thresholds,
    collect_set(quarter) as quarters,
    collect_set(window_agg) as window_aggs
FROM nwmd_metrics_by_location_v2 
GROUP BY configuration_name
""").show(truncate=False)

+------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+---------------------------------+------------------------------------------------------------------------+----------------+
|configuration_name|forecast_lead_time_bins                                                                                                                                               |water_years |thresholds                       |quarters                                                                |window_aggs     |
+------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+---------------------------------+------------------------------------------------------------------------+----------------+
|nwm30_medium_range|[P1DT

In [8]:
spark.sql("""
SELECT 
    configuration_name, water_year, threshold, quarter, window_agg, count(*)
FROM nwmd_metrics_by_location_v2
GROUP BY configuration_name, water_year, threshold, quarter, window_agg
ORDER BY configuration_name, water_year, threshold, quarter, window_agg
""").show(truncate=False)

+------------------+----------+---------+-------+----------+--------+
|configuration_name|water_year|threshold|quarter|window_agg|count(1)|
+------------------+----------+---------+-------+----------+--------+
|nwm30_medium_range|2025      |NULL     |NULL   |max       |81654   |
|nwm30_medium_range|2025      |NULL     |NULL   |mean      |81654   |
|nwm30_medium_range|2025      |NULL     |NULL   |min       |81654   |
|nwm30_medium_range|2025      |NULL     |2024-Q4|max       |80587   |
|nwm30_medium_range|2025      |NULL     |2024-Q4|mean      |80587   |
|nwm30_medium_range|2025      |NULL     |2024-Q4|min       |80587   |
|nwm30_medium_range|2025      |NULL     |2025-Q1|max       |80458   |
|nwm30_medium_range|2025      |NULL     |2025-Q1|mean      |80458   |
|nwm30_medium_range|2025      |NULL     |2025-Q1|min       |80458   |
|nwm30_medium_range|2025      |NULL     |2025-Q2|max       |81300   |
|nwm30_medium_range|2025      |NULL     |2025-Q2|mean      |81300   |
|nwm30_medium_range|

In [9]:
spark.stop()